# Use Case 6 — Embeddings

`text-embedding-3-small` converts text into a 1536-dimensional float vector.  
Similar texts have vectors that are geometrically close — measured by cosine similarity.

This notebook covers:

1. **Single embedding** — embed one string
2. **Batch embedding** — embed many strings in one API call
3. **Cosine similarity** — compare two vectors
4. **Semantic search** — rank candidates against a query

**Prerequisites:** Set `OPENAI_API_KEY` in the kernel environment or a `.env` file.

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

api_key = os.environ.get("OPENAI_API_KEY")
if not api_key:
    raise EnvironmentError("OPENAI_API_KEY is not set")

EMBED_MODEL = os.environ.get("EMBEDDING_MODEL", "text-embedding-3-small")
print(f"Using model: {EMBED_MODEL}")

In [ ]:
import numpy as np
from openai import OpenAI

client = OpenAI(api_key=api_key)

## 1. Single Embedding

In [ ]:
response = client.embeddings.create(
    model=EMBED_MODEL,
    input="The quick brown fox jumps over the lazy dog.",
)

vector = response.data[0].embedding
print(f"Dimensions : {len(vector)}")
print(f"First 5 dims: {vector[:5]}")
print(f"Tokens used: {response.usage.total_tokens}")

## 2. Batch Embedding

Pass a list of strings — one API call, much cheaper than looping.

In [ ]:
texts = [
    "Python is a high-level programming language.",
    "Machine learning enables computers to learn from data.",
    "The Eiffel Tower is in Paris.",
    "Neural networks are inspired by the human brain.",
    "France is a country in Western Europe.",
]

response = client.embeddings.create(model=EMBED_MODEL, input=texts)

# Re-sort by index to guarantee order matches input
sorted_data = sorted(response.data, key=lambda e: e.index)
vectors = [e.embedding for e in sorted_data]

print(f"Embedded {len(vectors)} texts")
print(f"Total tokens: {response.usage.total_tokens}")

## 3. Cosine Similarity

Two vectors are "similar" when their angle is small — cosine returns values in [-1, 1].

In [ ]:
def cosine_similarity(a: list[float], b: list[float]) -> float:
    va, vb = np.array(a), np.array(b)
    denom = np.linalg.norm(va) * np.linalg.norm(vb)
    return float(np.dot(va, vb) / denom) if denom else 0.0


# Build a similarity matrix for our 5 texts
n = len(texts)
print(f"{'':40s}" + "".join(f" T{i+1:2d}" for i in range(n)))
for i in range(n):
    label = texts[i][:38] + ".."
    scores = [cosine_similarity(vectors[i], vectors[j]) for j in range(n)]
    print(f"{label:40s}" + "".join(f" {s:4.2f}" for s in scores))

## 4. Semantic Search

Embed a query, then rank all candidates by cosine similarity descending.

In [ ]:
query = "What is a programming language?"

q_response = client.embeddings.create(model=EMBED_MODEL, input=query)
query_vec = q_response.data[0].embedding

results = sorted(
    [(text, cosine_similarity(query_vec, vec)) for text, vec in zip(texts, vectors)],
    key=lambda x: x[1],
    reverse=True,
)

print(f'Query: "{query}"\n')
for rank, (text, score) in enumerate(results, start=1):
    print(f"{rank}. [{score:.4f}] {text}")

## Key takeaways

- Always batch: one `embeddings.create(input=[...])` call for N texts is far cheaper than N calls.
- `text-embedding-3-small` produces 1536-dimensional unit-norm vectors — dot product == cosine similarity.
- Cosine similarity is position-independent: sentence length and phrasing don't confuse it.
- For production semantic search use a vector database (pgvector, Pinecone, Weaviate, etc.).